In [ ]:
print("ram ram")

In [ ]:
import os
import base64
import psycopg2
from typing import TypedDict, Annotated, Sequence
from langchain_core.tools import tool
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from litellm import completion
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import ToolNode
from dotenv import load_dotenv
load_dotenv()



# Keys Setup
os.environ["GEMINI_API_KEY"] = os.getenv("GEMINI_API_KEY")
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

CONSTRUCTION_DB_URL=os.getenv("CONSTRUCTION_DB_URL")
MANUFACTURING_DB_URL=os.getenv("MANUFACTURING_DB_URL")


In [ ]:

@tool
def fetch_construction_metrics(query_string: str) -> str:
    """Useful to query the construction analytics database for project metrics, timelines, or safety data. Input should be a SQL query."""
    try:
        conn = psycopg2.connect(CONSTRUCTION_DB_URL)
        cursor = conn.cursor()
        cursor.execute(query_string)
        records = cursor.fetchall()
        cursor.close()
        conn.close()
        return f"Database Results: {str(records)}"
    except Exception as e:
        return f"Database is empty or error occurred: {str(e)}"

@tool
def fetch_manufacturing_metrics(query_string: str) -> str:
    """Useful to query the manufacturing database for equipment efficiency, production logs, or downtime details. Input should be a SQL query."""
    try:
        conn = psycopg2.connect(MANUFACTURING_DB_URL)
        cursor = conn.cursor()
        cursor.execute(query_string)
        records = cursor.fetchall()
        cursor.close()
        conn.close()
        return f"Database Results: {str(records)}"
    except Exception as e:
        return f"Database is empty or error occurred: {str(e)}"

# Pack tools list
tools = [fetch_construction_metrics, fetch_manufacturing_metrics]
tool_node = ToolNode(tools)

In [ ]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], lambda x, y: x + y]
    chart_context: str

def encode_image(image_path: str) -> str:
    with open(image_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode('utf-8')

# Node 1: Extract Chart Visual Data via Gemini
def parse_chart_node(state: AgentState) -> dict:
    # Find if there's any file path passed in the latest message or context
    # Let's assume we pass the file path as part of state initiation
    print("🤖 [Gemini] Parsing chart details...")
    return {"chart_context": "Chart Analysis: [No chart image provided in state]"}

# Node 2: Groq Llama 3 Agent Decision Engine
def agent_core_node(state: AgentState) -> dict:
    print("⚡ [Groq] Deciding next steps or compiling final summary...")
    
    # We pass the accumulated messages and chart context to Groq using LiteLLM
    system_prompt = f"You are a dashboard analytics summary assistant. You have access to chart context: {state['chart_context']}. Use tools to query databases if needed to fulfill the user request, or provide a clean markdown summary if you have all information."
    
    formatted_messages = [{"role": "system", "content": system_prompt}]
    for msg in state["messages"]:
        if isinstance(msg, HumanMessage):
            formatted_messages.append({"role": "user", "content": msg.content})
        elif isinstance(msg, AIMessage):
            formatted_messages.append({"role": "assistant", "content": msg.content})
            
    # LiteLLM completion call supporting tool binding
    response = completion(
        model="groq/llama3-70b-8192",
        messages=formatted_messages,
        tools=[{
            "type": "function",
            "function": {
                "name": t.name,
                "description": t.description,
                "parameters": {"type": "object", "properties": {"query_string": {"type": "string"}}, "required": ["query_string"]}
            }
        } for t in tools]
    )
    
    # Handle response and translate to LangChain message structure
    message_content = response.choices[0].message.content or ""
    tool_calls = response.choices[0].message.tool_calls
    
    ai_message = AIMessage(content=message_content)
    if tool_calls:
        ai_message.tool_calls = [
            {"name": tc.function.name, "args": {"query_string": tc.function.arguments}, "id": tc.id, "type": "tool_call"}
            for tc in tool_calls
        ]
        
    return {"messages": [ai_message]}

# Conditional routing logic
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "execute_tools"
    return "end_workflow"


In [ ]:
workflow = StateGraph(AgentState)

# Add Nodes
workflow.add_node("parse_chart", parse_chart_node)
workflow.add_node("agent_core", agent_core_node)
workflow.add_node("execute_tools", tool_node)

# Set Dependencies
workflow.set_entry_point("parse_chart")
workflow.add_edge("parse_chart", "agent_core")

# Routing Loop
workflow.add_conditional_edges(
    "agent_core",
    should_continue,
    {
        "execute_tools": "execute_tools",
        "end_workflow": END
    }
)
workflow.add_edge("execute_tools", "agent_core")

# Compile
summary_agent = workflow.compile()


In [ ]:
summary_agent

In [ ]:
from langchain_core.messages import HumanMessage

def run_agent_test(test_name: str, input_type: str, file_path: str = None, user_query: str = None):
    """
    LangGraph Summary Agent ko testing scenarios ke sath debug aur run karne ka wrapper function.
    """
    print(f"\n{'='*15} 🚀 RUNNING TEST: {test_name.upper()} {'='*15}")
    
    # 1. User Message context build karna
    default_text = f"Please provide a summary for this {input_type}."
    final_query_text = user_query if user_query else default_text
    
    # 2. Initial state setup jise LangGraph dynamic node processing me access karega
    initial_state = {
        "messages": [HumanMessage(content=final_query_text)],
        "chart_context": ""  # Yeh variable 'parse_chart' node update karega
    }
    
    # 3. Custom payload handle karna agar chart file di gayi ho
    # (Aap is state context ko handle karne ke liye apne 'parse_chart_node' ko update kar sakte hain)
    if file_path:
        initial_state["file_path_input"] = file_path 
        print(f"📁 Target Document/Chart detected: {file_path}")

    print(f"💬 User Prompt sent to Agent: '{final_query_text}'\n")
    print("--- 🔄 Graph Node Execution Steps ---")
    
    # 4. LangGraph Stream running to keep track of active node processes
    final_state = None
    for output in dashboard_agent.stream(initial_state):
        for node_name, node_output in output.items():
            print(f"✅ Finished executing Node: [{node_name}]")
            # Tool calls ya execution tracking logs dekhne ke liye debugging logs:
            if "messages" in node_output and node_output["messages"]:
                last_msg = node_output["messages"][-1]
                if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
                    print(f"   🛠️ Agent invoked a DB tool: {last_msg.tool_calls[0]['name']}")
                    print(f"   📝 Generated SQL: {last_msg.tool_calls[0]['args'].get('query_string', '')}")
        final_state = output

    print("\n" + "="*50)
    print("📊 FINAL AGENT OUTPUT (Executive Summary):")
    print("="*50)
    
    # Final message stream syntax se message display print format
    if final_state:
        # LangGraph ke output dict se state data fetch karna
        last_node = list(final_state.keys())[-1]
        state_messages = final_state[last_node].get("messages", [])
        if state_messages:
            print(state_messages[-1].content)
        else:
            print("No output message captured. Check agent_core node configuration.")
    print(f"{'='*50}\n")


In [ ]:
run_agent_test(
    test_name="Chart Data Summary",
    input_type="chart",
    file_path="dashboard_leak.jpg"
)


In [ ]:
run_agent_test(
    test_name="Chart Data Summary",
    input_type="chart",
    file_path="dashboard_leak.jpg"
)


In [ ]:
run_agent_test(
    test_name="Chart Context + Construction Safety Tool Lookup",
    input_type="chart",
    file_path="construction_timeline.png",
    user_query="Compare the timeline trend visible in this chart with the safety incidents logged in the construction_ai database."
)
